# Recherche stratégie — copier les trades du Congrès (2014-2026)

**La démarche, dans l'ordre où on se pose les questions :**
1. **Les données** — la table canonique certifiée + les prix.
2. **L'événement disclosure** — que se passe-t-il après la *publication* d'un trade ? (et après la
   *transaction* elle-même, comme plafond théorique). C'est la question préalable à toute stratégie.
3. **La stratégie du brief Ramify** — copy-trading trade-based : entrée à la disclosure d'un achat,
   sortie à la disclosure de la vente du même membre, sinon +12 mois ; sélection annuelle top-K par
   Sharpe de la série de trades.
4. **Deux variantes issues de l'état de l'art** — les 12 leaders de parti · l'horizon court.
5. **V2 ETF sectoriels** — la version intégrable à l'univers Ramify.
6. **Verdict** — nos chiffres situés dans la littérature.

**Conventions (tout le notebook)** :
- Entrée toujours au **premier jour de bourse APRÈS la date d'événement** (J+1) — aucun look-ahead.
- **Coûts : 20 bps par transaction** (aller simple), soit 40 bps par aller-retour.
- Chaque trade = **une observation** ; excès mesuré **vs SPY** sur la même fenêtre.
- Contexte (état de l'art, `ETAT_DE_L_ART_STRATEGIES.md`) : le consensus académique post-STOCK Act est
  « pas d'alpha moyen » (Belmont 2022) ; les poches documentées : drift court post-disclosure
  (~90 bps/mois, temporaire), les 12 leaders de parti, les trades liés aux jalons législatifs.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

HERE = Path.cwd() if (Path.cwd() / "cache").exists() else Path.cwd() / "00. S3S4 en cours"
REPO = HERE.parent
CLEAN = REPO / "00_S1S2_donnees" / "data" / "clean" / "transactions_backtest_2014_2026.csv"
PRICES = HERE / "cache" / "prices_v2"

COST_BPS = 20            # coût par transaction, aller simple
HOLD_MAX = 252           # sortie forcée du brief : 12 mois = 252 jours de bourse
HORIZONS = {"1 sem": 5, "1 mois": 21, "3 mois": 63, "6 mois": 126, "12 mois": 252}

print("table  :", CLEAN.name, "| prix :", PRICES)

table  : transactions_backtest_2014_2026.csv | prix : /Users/lemairealice/Downloads/Jupiter/00. S3S4 en cours/cache/prices_v2


## 1. Les données

- **Transactions** : la table canonique certifiée (audit 2026-07-03) — sources officielles House+Sénat,
  100 % de l'index du Clerk traité, identité/parti/commissions à la date du trade, montants = milieu
  exact de fourchette, `ticker_yahoo` prêt pour les prix (renommages appliqués).
- **Prix** : cache v2 construit sur `ticker_yahoo` (yfinance, ajusté dividendes/splits), 2012→2026.
- On ne dé-duplique JAMAIS cette table (les lignes identiques = lots multi-comptes réels).

In [2]:
df = pd.read_csv(CLEAN, dtype={"doc_id": str}, parse_dates=["transaction_date", "disclosure_date"])
achats, ventes = df[df["direction"] == "buy"], df[df["direction"] == "sell"]
print(f"{len(df):,} transactions | {df['bioguide_id'].nunique()} membres | "
      f"{df['ticker_yahoo'].nunique():,} tickers")
print(f"achats {len(achats):,} | ventes {len(ventes):,} | fenêtre filed "
      f"{df['disclosure_date'].min().date()} → {df['disclosure_date'].max().date()}")
print(f"délai médian transaction → publication : {df['lag_days'].median():.0f} jours")

134,464 transactions | 372 membres | 4,618 tickers
achats 68,250 | ventes 66,214 | fenêtre filed 2014-01-02 → 2026-07-02
délai médian transaction → publication : 27 jours


In [3]:
# Panel de prix : une colonne par ticker, calendrier de bourse = celui de SPY.
# Garde-fou anti-corruption : Yahoo renvoie parfois des séries cassées (prix négatifs, sauts ×1000) —
# toute série avec un prix < 0,10 $ ou un saut quotidien > 300 % est écartée (61 tickers, ~0,3 % des trades).
def load_price(tk):
    f = PRICES / f"{tk}.csv"
    if not f.exists() or f.stat().st_size < 200:
        return None
    s = pd.read_csv(f, parse_dates=["Date"]).set_index("Date")["close"]
    if s.min() < 0.10 or s.pct_change().abs().max() > 3.0:
        return None
    return s

spy = load_price("SPY")
CAL = spy.index                                   # calendrier maître
tickers = sorted(set(df["ticker_yahoo"].dropna()))
series = {tk: s for tk in tickers if (s := load_price(tk)) is not None}
px = pd.DataFrame(series).reindex(CAL)
spy = spy.reindex(CAL)

couverture = df["ticker_yahoo"].isin(px.columns)
print(f"panel prix : {px.shape[1]:,} tickers × {len(CAL):,} jours "
      f"({CAL.min().date()} → {CAL.max().date()})")
print(f"trades couverts par un prix : {couverture.sum():,} / {len(df):,} "
      f"({couverture.mean():.1%}) — les non couverts sont surtout des délistés sans successeur")

panel prix : 3,289 tickers × 3,645 jours (2012-01-03 → 2026-07-02)
trades couverts par un prix : 118,061 / 134,464 (87.8%) — les non couverts sont surtout des délistés sans successeur


## 2. L'événement disclosure — la question préalable

- **Deux dates par trade** : la `transaction_date` (quand l'élu a agi — information parfaite,
  inaccessible à un copieur) et la `disclosure_date` (quand c'est publié — la seule date jouable).
- **Ce qu'on mesure** : l'excès de rendement vs SPY, en entrant au premier jour de bourse après
  chacune des deux dates, à 5 horizons (1 sem → 12 mois). Achats et ventes séparés.
- **Pourquoi c'est LA question** : si même à la date de transaction il n'y a rien, tout l'aval est
  borné. Et la littérature récente (Lazzaretto 2024, Pyun 2025) dit que l'essentiel du signal
  *copiable* serait un **drift court APRÈS la publication** — invisible à horizon 12 mois.
- *Lecture honnête des t-stats : les fenêtres de trades se chevauchent (corrélation entre
  observations) → les t affichés sont optimistes ; on les lit comme des ordres de grandeur.*

In [4]:
# Excès de rendement vectorisé : entrée à J+1 après `dates`, sortie h jours de bourse plus tard.
PXV = px.to_numpy()
SPYV = spy.to_numpy()
COL = {tk: i for i, tk in enumerate(px.columns)}

def batch_excess(tks, dates, h):
    i0 = CAL.searchsorted(pd.DatetimeIndex(dates), side="right")   # J+1
    i1 = i0 + h
    cols = np.array([COL.get(t, -1) for t in tks])
    ok = (cols >= 0) & (i1 < len(CAL))
    out = np.full(len(tks), np.nan)
    r, c0, c1 = cols[ok], i0[ok], i1[ok]
    p0, p1 = PXV[c0, r], PXV[c1, r]
    b0, b1 = SPYV[c0], SPYV[c1]
    valid = ~(np.isnan(p0) | np.isnan(p1)) & (p0 > 0) & (b0 > 0)
    res = np.full(ok.sum(), np.nan)
    res[valid] = (p1[valid] / p0[valid] - 1) - (b1[valid] / b0[valid] - 1)
    out[ok] = res
    return out

In [5]:
# Tous les achats, tous les horizons, depuis les DEUX dates.
def table_evenement(sub):
    rows = []
    for nom, h in HORIZONS.items():
        for date_col, etiquette in [("transaction_date", "transaction (plafond)"),
                                    ("disclosure_date", "publication (jouable)")]:
            x = batch_excess(sub["ticker_yahoo"].to_numpy(), sub[date_col].to_numpy(), h)
            x = x[~np.isnan(x)]
            t = x.mean() / (x.std() / np.sqrt(len(x))) if len(x) > 1 else np.nan
            rows.append({"horizon": nom, "depuis": etiquette, "n": len(x),
                         "excès moyen": f"{x.mean():+.2%}", "médiane": f"{np.median(x):+.2%}",
                         "% gagnants": f"{(x > 0).mean():.0%}", "t": round(t, 2)})
    return pd.DataFrame(rows)

tab_achats = table_evenement(achats)
print("ACHATS — excès vs SPY (entrée J+1)")
print(tab_achats.to_string(index=False))

ACHATS — excès vs SPY (entrée J+1)
horizon                depuis     n excès moyen médiane % gagnants     t
  1 sem transaction (plafond) 58925      -0.01%  -0.06%        49% -0.50
  1 sem publication (jouable) 58902      -0.04%  -0.07%        49% -2.25
 1 mois transaction (plafond) 58881      -0.18%  -0.20%        48% -5.13
 1 mois publication (jouable) 58504      +0.07%  -0.23%        48%  1.89
 3 mois transaction (plafond) 58104      -0.27%  -0.73%        47% -4.15
 3 mois publication (jouable) 57628      -0.36%  -0.75%        47% -5.67
 6 mois transaction (plafond) 56532      -0.76%  -1.70%        45% -7.99
 6 mois publication (jouable) 56244      -0.74%  -1.78%        45% -7.76
12 mois transaction (plafond) 54135      -0.58%  -3.69%        44% -3.62
12 mois publication (jouable) 53109      -0.35%  -3.53%        44% -2.09


In [6]:
tab_ventes = table_evenement(ventes)
print("VENTES — excès vs SPY des titres VENDUS (entrée J+1 ; négatif = la vente était bien avisée)")
print(tab_ventes.to_string(index=False))

VENTES — excès vs SPY des titres VENDUS (entrée J+1 ; négatif = la vente était bien avisée)
horizon                depuis     n excès moyen médiane % gagnants     t
  1 sem transaction (plafond) 57024      +0.03%  -0.03%        49%  1.27
  1 sem publication (jouable) 56973      -0.14%  -0.12%        48% -7.42
 1 mois transaction (plafond) 56968      -0.11%  -0.20%        49% -2.87
 1 mois publication (jouable) 56396      -0.12%  -0.28%        48% -3.00
 3 mois transaction (plafond) 56046      -0.31%  -0.87%        47% -4.18
 3 mois publication (jouable) 55615      -0.43%  -0.94%        47% -6.32
 6 mois transaction (plafond) 54461      -0.66%  -1.89%        45% -6.05
 6 mois publication (jouable) 54168      -0.67%  -1.99%        45% -5.94
12 mois transaction (plafond) 51827      -0.69%  -3.47%        44% -3.48
12 mois publication (jouable) 51121      -0.55%  -3.61%        44% -2.55


### Lecture — la question est tranchée

- **Il n'y a pas d'information moyenne dans les trades, même à la date de transaction** : les achats
  sous-performent SPY à tous les horizons ≥ 1 mois, y compris au plafond théorique (−0,76 % à 6 mois,
  t −8,0 ; −0,58 % à 12 mois). L'intuition de départ était juste : si le plafond est vide, la version
  jouable ne peut pas faire mieux — et c'est exactement ce que trouve la littérature (Belmont 2022 :
  −26 bps à 6 mois ; Chen & Sacerdote : les élus achètent après le pic d'attention).
- **Une seule trace positive : +0,07 % le mois qui suit la PUBLICATION** (t 1,9) — le « drift
  post-disclosure » documenté par Abdurakhmonov (+12-21 bps) et Lazzaretto. Ordre de grandeur : ~7 bps,
  soit **6× moins que nos coûts** d'aller-retour (40 bps). Réel, mais pas exploitable en moyenne.
- **Les ventes ne disent rien de plus** : les titres vendus sous-performent aussi (−0,55 % à 12 mois) —
  du même ordre que les achats → **aucun spread directionnel** (la leçon Ziobrowski-House : quand tout
  ce que touche le Congrès « sous-performe » pareil, c'est du style, pas du signal).
- **Attention au benchmark** : SPY 2014-2026 = les méga-caps. Un panier diversifié non-mégacap
  sous-performe mécaniquement ce benchmark (Hanousek : le signe peut basculer selon la référence).
  Les −0,3/−0,7 % ci-dessus mesurent donc largement un tilt de style, pas une « bêtise » du Congrès.
- La médiane (−3,5 % à 12 mois) très en dessous de la moyenne (−0,35 %) = **loterie à queue droite** :
  quelques gros gagnants rares portent la moyenne. C'est cette queue que la sélection de membres va
  tenter de capter — section suivante.

## 3. La stratégie du brief Ramify — copy-trading trade-based

**Les règles, telles que le brief les fixe :**
- **Entrée** : dès qu'une disclosure « Purchase » d'un membre suivi est publiée → position ouverte au
  premier jour de bourse après la `disclosure_date`.
- **Sortie** : à la disclosure de la première « Sale » du même membre sur ce ticker ; sinon **sortie
  forcée 12 mois** après l'entrée. Chaque trade = une observation, entrée/sortie explicites.
- **Sélection annuelle** : fin d'année Y, on choisit les **K membres** (K ∈ {4, 6, 8, 10}) à suivre pour
  Y+1 ; éligible = **≥ 10 trades clôturés** sur la période d'évaluation ; critère = **Sharpe de la série
  de trades réalisés** ; option du brief : **≥ la moitié des K** dans une commission clé
  (Finance/Défense/Renseignement). Les positions des non-reconduits courent jusqu'à leur sortie.
- Coûts : 40 bps par aller-retour ; excès mesuré vs SPY sur la même fenêtre.

In [7]:
# Appariement achat → première vente ultérieure du même (membre, ticker), sur les dates de PUBLICATION.
b = achats[couverture & achats["disclosure_date"].notna()].copy() if False else None
b = achats[achats["ticker_yahoo"].isin(px.columns)].copy()
s = ventes[["bioguide_id", "ticker_yahoo", "disclosure_date"]].dropna().sort_values("disclosure_date")
b = b.sort_values("disclosure_date").reset_index(drop=True)

paires = pd.merge_asof(
    b[["bioguide_id", "ticker_yahoo", "disclosure_date"]].reset_index(),
    s.rename(columns={"disclosure_date": "sortie_vente"}),
    left_on="disclosure_date", right_on="sortie_vente",
    by=["bioguide_id", "ticker_yahoo"], direction="forward", allow_exact_matches=False)
b["sortie_vente"] = paires.set_index("index")["sortie_vente"]

part_vente = b["sortie_vente"].notna().mean()
print(f"{len(b):,} achats backtestables | sortie déclenchée par une vente du membre : {part_vente:.0%} "
      f"| sinon sortie forcée +12 mois")

59,998 achats backtestables | sortie déclenchée par une vente du membre : 80% | sinon sortie forcée +12 mois


In [8]:
# Rendement net par trade : entrée J+1 après l'achat publié, sortie J+1 après la vente publiée
# (sinon entrée + 252 j de bourse). Titres délistés en détention : dernier prix connu (limite assumée).
PXF = px.ffill().to_numpy()

def trade_excess_net(tks, d_in, d_out):
    i0 = CAL.searchsorted(pd.DatetimeIndex(d_in), side="right")
    i1 = np.where(pd.isna(d_out),
                  i0 + HOLD_MAX,
                  CAL.searchsorted(pd.DatetimeIndex(pd.Series(d_out).fillna(CAL[-1])), side="right"))
    i1 = np.minimum(np.maximum(i1, i0 + 1), len(CAL) - 1)
    cols = np.array([COL.get(t, -1) for t in tks])
    ok = (cols >= 0) & (i0 < len(CAL) - 1)
    out = np.full(len(tks), np.nan)
    r, c0, c1 = cols[ok], i0[ok], np.asarray(i1)[ok]
    p0, p1 = PXV[c0, r], PXF[c1, r]
    b0, b1 = SPYV[c0], SPYV[c1]
    valid = ~(np.isnan(p0) | np.isnan(p1)) & (p0 > 0)
    res = np.full(int(ok.sum()), np.nan)
    res[valid] = (p1[valid] / p0[valid] - 1) - (b1[valid] / b0[valid] - 1) - 2 * COST_BPS / 1e4
    out[ok] = res
    return out, i1

b["exces_net"], b["_i_sortie"] = trade_excess_net(
    b["ticker_yahoo"].to_numpy(), b["disclosure_date"].to_numpy(), b["sortie_vente"].to_numpy())
b["date_sortie"] = CAL[np.minimum(b["_i_sortie"], len(CAL) - 1)]
bt = b.dropna(subset=["exces_net"]).copy()
print(f"{len(bt):,} trades avec rendement net calculable | excès net moyen {bt['exces_net'].mean():+.2%} "
      f"| médiane {bt['exces_net'].median():+.2%} | % gagnants {(bt['exces_net'] > 0).mean():.0%}")

58,910 trades avec rendement net calculable | excès net moyen +0.56% | médiane -2.27% | % gagnants 43%


### Les track records individuels — qui « mérite » d'être suivi ?

- Pour chaque membre : la **série de ses trades clôturés** (excès net par trade) → **Sharpe de la
  série** (moyenne / écart-type des excès), le critère du brief.
- On ne regarde ici que le plein-échantillon, pour voir la dispersion ; la sélection HONNÊTE
  (walk-forward, sans regarder le futur) vient juste après.

In [9]:
tr = (bt.groupby(["bioguide_id", "member_name"])["exces_net"]
        .agg(n="count", exces_moyen="mean", ecart_type="std")
        .query("n >= 10").reset_index())
tr["sharpe_trades"] = tr["exces_moyen"] / tr["ecart_type"]
tr = tr.sort_values("sharpe_trades", ascending=False)
print(f"{len(tr)} membres éligibles (≥10 trades clôturés) sur tout 2014-2026")
print(tr.head(10)[["member_name", "n", "exces_moyen", "sharpe_trades"]]
      .assign(exces_moyen=lambda d: d["exces_moyen"].map("{:+.2%}".format),
              sharpe_trades=lambda d: d["sharpe_trades"].round(2)).to_string(index=False))

196 membres éligibles (≥10 trades clôturés) sur tout 2014-2026
          member_name   n exces_moyen  sharpe_trades
              Ed Case  10      +8.01%           3.41
      Victoria Spartz  22     +15.50%           1.26
    Daniel S Sullivan  35     +45.69%           0.67
   Trey Hollingsworth  16      +7.87%           0.59
         Ashley Moody  16     +52.90%           0.55
         Kathy Castor  52     +14.65%           0.50
    Frank A. LoBiondo  18      +8.40%           0.49
         Justin Amash  18     +10.61%           0.46
            Dave Camp 213      +9.82%           0.46
Kelly Louise Morrison  10     +21.98%           0.45


In [10]:
# Walk-forward : sélection au 31/12 de chaque année sur les trades CLÔTURÉS avant cette date,
# suivi des achats publiés l'année suivante. Aucune information future n'entre dans la sélection.
def membre_cle(annee):
    fen = bt[bt["disclosure_date"].dt.year == annee]
    return fen.groupby("bioguide_id")["committees_key_flag"].agg(
        lambda s: s.astype(str).str.lower().eq("true").mean() >= 0.5)

def selection(annee, K, filtre_commission):
    histo = bt[bt["date_sortie"] <= pd.Timestamp(f"{annee}-12-31")]
    g = histo.groupby("bioguide_id")["exces_net"].agg(n="count", m="mean", sd="std")
    g = g[(g["n"] >= 10) & (g["sd"] > 0)]
    g["sharpe"] = g["m"] / g["sd"]
    classement = g.sort_values("sharpe", ascending=False).index.tolist()
    if not filtre_commission:
        return classement[:K]
    cle = membre_cle(annee)
    pick, n_cle = [], 0
    for bio in classement:
        if len(pick) == K:
            break
        pick.append(bio); n_cle += bool(cle.get(bio, False))
    # impose ≥ K/2 membres « commission clé » en remplaçant les queues non-clés
    if n_cle < K / 2:
        reserves = [x for x in classement if x not in pick and cle.get(x, False)]
        for i in range(len(pick) - 1, -1, -1):
            if n_cle >= K / 2 or not reserves:
                break
            if not cle.get(pick[i], False):
                pick[i] = reserves.pop(0); n_cle += 1
    return pick

ANNEES = range(2015, 2027)
resultats = []
for K in (4, 6, 8, 10):
    for filtre in (False, True):
        exces_annuels = []
        for annee in ANNEES:
            suivis = selection(annee - 1, K, filtre)
            t_an = bt[(bt["disclosure_date"].dt.year == annee) & bt["bioguide_id"].isin(suivis)]
            if len(t_an):
                exces_annuels.append(t_an["exces_net"].mean())
        x = pd.Series(exces_annuels)
        resultats.append({"K": K, "filtre commissions": "oui" if filtre else "non",
                          "années": len(x), "excès/trade moyen": f"{x.mean():+.2%}",
                          "écart-type": f"{x.std():.2%}", "années > 0": f"{(x > 0).mean():.0%}",
                          "t": round(x.mean() / (x.std() / np.sqrt(len(x))), 2) if len(x) > 1 else np.nan})
grille = pd.DataFrame(resultats)
print("STRATÉGIE DU BRIEF — walk-forward, net de coûts, excès moyen PAR TRADE suivi, par année")
print(f"({len(grille)} configurations essayées — à garder en tête en lisant les t)")
print(grille.to_string(index=False))

STRATÉGIE DU BRIEF — walk-forward, net de coûts, excès moyen PAR TRADE suivi, par année
(8 configurations essayées — à garder en tête en lisant les t)
 K filtre commissions  années excès/trade moyen écart-type années > 0    t
 4                non      10            +8.29%     11.96%        70% 2.19
 4                oui      12            +0.54%      7.90%        50% 0.24
 6                non      10            +5.12%     11.09%        70% 1.46
 6                oui      12            +2.99%     10.57%        50% 0.98
 8                non      10            +6.45%     12.21%        70% 1.67
 8                oui      12            +4.50%      8.37%        67% 1.86
10                non      11            +2.75%     15.73%        73% 0.58
10                oui      12            +3.82%      5.27%        83% 2.52


### Lecture V1 — positif, mais pas une preuve

- L'excès net moyen de TOUS les trades suivis est **+0,56 % par trade** (médiane −2,27 %, 43 % de
  gagnants) : la queue droite, encore.
- La grille top-K donne des excès **positifs dans les 8 configurations** (+0,5 % à +8,3 % par trade)
  avec des t entre 0,2 et 2,5. Mais : **8 essais** (le meilleur t attendu par pur hasard sur 8 essais
  ≈ 2), **instabilité forte entre configurations voisines** (K=10 : t 0,58 sans filtre → 2,52 avec),
  et le rappel Grinold-Kahn de nos travaux passés (breadth IC ≈ 0,02 → un excès > 3-4 %/an est suspect
  par construction). → **aucune configuration ne constitue une preuve d'alpha**.
- C'est la même conclusion que le backtest RAMIFY_V1 sur l'ancienne donnée (meilleur t = 0,88) — la
  donnée corrigée (ventes pré-2018 récupérées, vrais montants, meilleurs prix) donne des niveaux un peu
  plus élevés mais toujours dans la zone du hasard de sélection.
- Les « meilleurs » track records individuels (Sullivan +46 %/trade, Moody +53 %) sont des queues
  réelles mais tardivement détectables : le walk-forward ne les attrape qu'après coup — exactement le
  problème skill vs luck (Mauboussin) que le brief anticipait.

## 4. Deux variantes issues de l'état de l'art

**(a) Les 12 leaders de parti** — Wei & Zhou (NBER w34524) : l'alpha serait concentré sur les 12 postes
de leadership de PARTI (Speaker, floor leaders, whips, conference chairs — PAS les chairs de
commissions), et il **persisterait aux dates de disclosure** (~+10,5 %/an brut). Aucun produit ne le
copie. On teste : copier TOUS les achats publiés d'un membre pendant qu'il occupe un de ces postes.

**(b) L'horizon court** — Lazzaretto 2024 : le drift post-publication (~90 bps le 1er mois) est
**temporaire**. Une sortie à 12 mois peut le diluer entièrement. On re-teste la copie avec sortie à
**1 mois** au lieu de 12.

In [11]:
# (a) Leaders de parti — table vérifiée (postes × mandats), jointure point-in-time à la disclosure.
LEAD = pd.read_csv(HERE / "leadership_2014_2026.csv", parse_dates=["debut", "fin"])
lb = b.merge(LEAD[["bioguide_id", "debut", "fin"]], on="bioguide_id", how="inner")
lb = lb[(lb["disclosure_date"] >= lb["debut"]) & (lb["disclosure_date"] <= lb["fin"])]
lb = lb.drop_duplicates(subset=["bioguide_id", "ticker_yahoo", "disclosure_date"])
print(f"{len(lb):,} achats publiés PAR un leader en poste ({lb['bioguide_id'].nunique()} leaders traders)")

lignes = []
for nom, h in [("1 mois", 21), ("3 mois", 63), ("12 mois", 252)]:
    x = batch_excess(lb["ticker_yahoo"].to_numpy(), lb["disclosure_date"].to_numpy(), h)
    x = x[~np.isnan(x)] - 2 * COST_BPS / 1e4
    t = x.mean() / (x.std() / np.sqrt(len(x))) if len(x) > 1 else np.nan
    lignes.append({"horizon": nom, "n": len(x), "excès net moyen": f"{x.mean():+.2%}",
                   "médiane": f"{np.median(x):+.2%}", "% gagnants": f"{(x > 0).mean():.0%}",
                   "t": round(t, 2)})
print(pd.DataFrame(lignes).to_string(index=False))

745 achats publiés PAR un leader en poste (6 leaders traders)
horizon   n excès net moyen médiane % gagnants     t
 1 mois 742          -1.25%  -1.71%        38% -3.62
 3 mois 742          -0.78%  -3.24%        40% -1.05
12 mois  78          +5.40%  -0.11%        50%  1.22


In [12]:
# (b) Horizon court — TOUS les achats publiés (pas de sélection), sortie 1 mois vs 12 mois.
lignes = []
for nom, h in [("1 sem", 5), ("1 mois", 21), ("3 mois", 63), ("12 mois", 252)]:
    x = batch_excess(b["ticker_yahoo"].to_numpy(), b["disclosure_date"].to_numpy(), h)
    x = x[~np.isnan(x)] - 2 * COST_BPS / 1e4
    t = x.mean() / (x.std() / np.sqrt(len(x))) if len(x) > 1 else np.nan
    lignes.append({"sortie à": nom, "n": len(x), "excès net moyen": f"{x.mean():+.2%}",
                   "annualisé ~": f"{x.mean() * 252 / h:+.1%}", "t": round(t, 2)})
print("TOUS LES ACHATS à la publication — le drift court survit-il aux coûts ?")
print(pd.DataFrame(lignes).to_string(index=False))
print("\n(rappel : 40 bps de coûts par aller-retour pèsent d'autant plus que l'horizon est court)")

TOUS LES ACHATS à la publication — le drift court survit-il aux coûts ?
sortie à     n excès net moyen annualisé ~      t
   1 sem 58902          -0.44%      -22.1% -25.48
  1 mois 58504          -0.33%       -4.0%  -8.88
  3 mois 57628          -0.76%       -3.0% -12.04
 12 mois 53109          -0.75%       -0.8%  -4.48

(rappel : 40 bps de coûts par aller-retour pèsent d'autant plus que l'horizon est court)


### Lecture variantes

- **Leaders de parti : ne se réplique PAS chez nous.** 745 achats publiés par seulement 6 leaders
  traders sur 2014-2026 ; à 1 mois : **−1,25 %** net (t −3,6) ; à 12 mois : +5,4 % mais n = 78 trades
  seulement (l'essentiel des trades de leaders est trop récent pour avoir 12 mois d'historique) et
  t = 1,2. L'alpha leaders de Wei & Zhou (1995-2021, ~50 leaders) ne survit pas sur notre fenêtre —
  cohérent avec notre essai précédent (effet porté par un seul membre).
- **Horizon court : mort après coûts.** Le drift brut de +7 bps à 1 mois devient **−0,33 %** net
  (t −8,9) ; à 1 semaine, −0,44 % (les 40 bps d'aller-retour dominent tout). Il n'existe pas de
  stratégie de masse « copier vite à la publication » — le drift documenté par la littérature est réel
  mais trop petit, et conditionné (juridiction de comité, trades « spéculatifs ») pour être capté en
  moyenne non conditionnée.

## 5. V2 — la version ETF sectoriels (l'univers Ramify)

- Mêmes trades que la meilleure configuration V1, mais l'instrument devient **l'ETF SPDR du secteur**
  du titre (`etf_proxy` de la table : NVDA → XLK, etc.). Entrée/sortie identiques.
- Ce que la littérature et notre essai précédent prédisent : le peu de signal du Congrès est
  **spécifique au titre** — la substitution sectorielle devrait le diluer fortement.

In [13]:
K_STAR, FILTRE_STAR = 8, False   # à ajuster après lecture de la grille V1
etf_ok = b["etf_proxy"].isin(px.columns)
lignes = []
for annee in ANNEES:
    suivis = selection(annee - 1, K_STAR, FILTRE_STAR)
    t_an = b[(b["disclosure_date"].dt.year == annee) & b["bioguide_id"].isin(suivis) & etf_ok]
    if not len(t_an):
        continue
    x_act, _ = trade_excess_net(t_an["ticker_yahoo"].to_numpy(), t_an["disclosure_date"].to_numpy(),
                                t_an["sortie_vente"].to_numpy())
    x_etf, _ = trade_excess_net(t_an["etf_proxy"].to_numpy(), t_an["disclosure_date"].to_numpy(),
                                t_an["sortie_vente"].to_numpy())
    lignes.append({"année": annee, "n": int(np.isfinite(x_etf).sum()),
                   "V1 actions": np.nanmean(x_act), "V2 ETF": np.nanmean(x_etf)})
v2 = pd.DataFrame(lignes)
print(f"V1 vs V2 (K={K_STAR}, mêmes trades, mêmes dates) — excès net moyen par trade et par année")
print(v2.assign(**{c: v2[c].map("{:+.2%}".format) for c in ("V1 actions", "V2 ETF")}).to_string(index=False))
print(f"\nMoyenne 2015-2026 : V1 {v2['V1 actions'].mean():+.2%} | V2 {v2['V2 ETF'].mean():+.2%} "
      f"→ dilution {v2['V1 actions'].mean() - v2['V2 ETF'].mean():+.2%} par trade")

V1 vs V2 (K=8, mêmes trades, mêmes dates) — excès net moyen par trade et par année
 année   n V1 actions V2 ETF
  2015 591     +3.61% +3.80%
  2016 135     -1.53% +1.33%
  2017 302    +26.17% -2.04%
  2018 179     +4.68% -3.54%
  2019 163     +9.57% +0.96%
  2020 214     +5.41% +1.37%
  2021  32     -8.87% +5.48%
  2022  89     +4.82% -0.12%
  2023   3     -6.80% +1.88%
  2025  16    +27.67% +1.26%

Moyenne 2015-2026 : V1 +6.47% | V2 +1.04% → dilution +5.43% par trade


## 6. Verdict

| Question | Réponse (2014-2026, donnée canonique, net de 40 bps A/R) |
|---|---|
| De l'info dans les trades, au plafond (`transaction_date`) ? | **Non** — achats −0,18 % (1 m) à −0,76 % (6 m) vs SPY, t < −4 |
| Un drift copiable à la publication ? | Trace réelle (+7 bps le 1er mois, t 1,9) mais **6× sous les coûts** |
| La stratégie du brief (top-K Sharpe, sortie vente/12 m) ? | Excès positifs (+0,5 à +8,3 %/trade selon config) mais **non robustes** : 8 essais, instabilité inter-K, queue droite — pas une preuve d'alpha |
| Les 12 leaders de parti (Wei & Zhou) ? | **Ne se réplique pas** : −1,25 % à 1 mois (t −3,6), n=6 leaders traders |
| La V2 ETF sectorielle ? | **Dilution confirmée** : +6,5 % → +1,0 % par trade (−5,4 pp) — le peu de signal est spécifique au titre |

**Ce que ça veut dire pour Ramify** :
- Notre résultat réplique le consensus académique (Belmont 2022, NANC/GOP en argent réel) : **pas
  d'alpha moyen copiable** dans les trades du Congrès post-STOCK Act ; ce qui reste est une loterie à
  queue droite qu'une sélection ex-ante ne capte pas de façon robuste.
- Un produit « Congrès » se défend comme **bêta thématique transparent** (à la NANC : +2,7 pp/an vs SPY
  depuis 2023, mais −3,8 pp vs QQQ — du style, pas du talent), pas comme une promesse de battre le
  marché. La V2 ETF dé-risque mais n'ajoute rien.
- **Pistes restantes non testées ici** (conditionnements documentés par la littérature, cf.
  `ETAT_DE_L_ART_STRATEGIES.md`) : juridiction de comité × industrie (+50 bps à l'annonce), trades ×
  jalons législatifs (Li 2025 : +3-5 %), conviction par taille (« Best Ideas »). Matière pour une v2 de
  ce notebook si souhaité.
- **Risque réglementaire** : le HONEST Act (ban du trading des élus) a passé la commission sénatoriale
  en 2025 — toute mise en production vivrait sous ce risque d'extinction du signal.

**Limites assumées de cette v1** : t-stats simples (fenêtres chevauchantes → t optimistes — ils
n'étaient déjà pas significatifs) ; couverture prix 87,8 % des trades (délistés sans successeur exclus
→ légère borne haute) ; sizing equal-weight par trade (le brief ne précise pas la pondération).